<a href="https://colab.research.google.com/github/tamil2223/ML/blob/master/tool_calling_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning a Small LLM for Tool Routing

**Goal:** Train a 1B parameter model to correctly route customer queries to the right tool.

| Query Type | Tool | Example |
|---|---|---|
| Order/shipping/delivery questions | `track_order` | "Where is my order ORD-8821?" |
| Product discovery/recommendations | `search_products` | "Suggest a laptop under 60k for coding" |

**What we'll show:**
1. The base model **cannot** do structured tool routing
2. We create a training dataset (~80 examples)
3. Fine-tune with Unsloth + LoRA in ~10 minutes on free Colab T4
4. The fine-tuned model **nails it** — correct tool, correct params, valid JSON

## Step 0: Install Unsloth

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## Step 1: Load the Base Model

We use **Llama 3.2 1B Instruct** in 4-bit quantization.

Why 1B? It's small enough to train fast on free Colab, and small enough that it will clearly **fail** at structured tool calling — making the before/after dramatic.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)
print("Model loaded.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.


Model loaded.


## Step 2: Define Our Tools and Test the Base Model

We define two tools that a customer support system would use.
Then we ask the base model to route queries — and watch it fail.

In [ ]:
SYSTEM_PROMPT = """You are a customer support assistant. You have access to exactly two tools:

1. track_order: Use when the customer asks about order status, shipping, delivery, or tracking.
   Parameters: {"order_id": string}

2. search_products: Use when the customer asks about finding products, recommendations, comparisons, or availability.
   Parameters: {"query": string, "max_budget": number or null}

RESPOND WITH ONLY a JSON object in this exact format, nothing else:
{"tool": "<tool_name>", "params": {<parameters>}}

Do not explain. Do not add any text before or after the JSON."""

TEST_QUERIES = [
    # Says "order" but wants to BUY something → search_products
    "I want to order a new gaming mouse. What do you have?",
    # Says "product" but asking about delivery → track_order
    "The product I ordered as ORD-4412 was supposed to come today. What's going on?",
    # "looking for" sounds like search, but it's a lost package → track_order
    "I'm looking for my package ORD-8891, it seems to have gone missing",
    # No order ID at all, model must still pick track_order
    "I bought something from you guys 3 days ago and haven't heard anything since. Can someone check?",
    # Multi-intent, primary is tracking → track_order
    "Where the hell is ORD-5523? Also, do you have any human I can connect to?",
    # Defective product, wants replacement → search_products
    "The keyboard I got is defective, keys are sticking. Show me alternatives under 3000",
    # Informal, no fluff → track_order
    "bro just tell me if ORD-2290 is coming today or not",
    # Vague, no category → search_products
    "my budget is 10000, surprise me with something cool",
    # Number looks like order ID but it's a price → search_products
    "I saw a monitor listed at 24999 on your site. Got anything similar but cheaper?",
    # Order ID buried in a rambling story → track_order
    "So basically my wife's birthday is tomorrow and I ordered her gift last week, order ORD-7763, and if this doesn't show up on time I will post this on social media.",
]

EXPECTED = [
    "search_products",
    "track_order",
    "track_order",
    "track_order",
    "track_order",
    "search_products",
    "track_order",
    "search_products",
    "search_products",
    "track_order",
]

print("Tools and test queries defined.")

Tools and test queries defined.


In [ ]:
# Enable inference mode for the base model
model.eval()

def run_inference(model, tokenizer, user_query):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(
        [text],
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=True,
            use_cache=False,
        )
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()


print("=" * 70)
print("BASE MODEL RESPONSES (before fine-tuning)")
print("=" * 70)
for query, expected_tool in zip(TEST_QUERIES, EXPECTED):
    response = run_inference(model, tokenizer, query)
    print(f"\n{'─' * 60}")
    print(f"QUERY:    {query[:80]}")
    print(f"EXPECTED: {expected_tool}")
    print(f"GOT:      {response[:200]}")

    # Check if it's valid JSON with correct tool
    import json
    try:
        parsed = json.loads(response)
        if parsed.get("tool") == expected_tool:
            print("RESULT:   ✅ Correct (lucky!)")
        else:
            print(f"RESULT:   ❌ Wrong tool or format")
    except json.JSONDecodeError:
        print("RESULT:   ❌ Not valid JSON")

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASE MODEL RESPONSES (before fine-tuning)


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


────────────────────────────────────────────────────────────
QUERY:    I want to order a new gaming mouse. What do you have?
EXPECTED: search_products
GOT:      {"tool": "search_products", "params": {"query": "gaming mouse", "max_budget": null}}
RESULT:   ✅ Correct (lucky!)


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


────────────────────────────────────────────────────────────
QUERY:    bro just tell me if ORD-2290 is coming today or not
EXPECTED: track_order
GOT:      {"tool": "track_order", "params": {"order_id": "ORD-2290"}}
RESULT:   ✅ Correct (lucky!)


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    my budget is 10000, surprise me with something cool
EXPECTED: search_products
GOT:      {"tool": "search_products", "params": {"query": "cool products under $10000", "max_budget": "10000"}}
RESULT:   ✅ Correct (lucky!)


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    I saw a monitor listed at 24999 on your site. Got anything similar but cheaper?
EXPECTED: search_products
GOT:      {"tool": "search_products", "params": {"query": "monitor cheaper", "max_budget": "0"}}
RESULT:   ✅ Correct (lucky!)

────────────────────────────────────────────────────────────
QUERY:    So basically my wife's birthday is tomorrow and I ordered her gift last week, or
EXPECTED: track_order
GOT:      {"tool": "track_order", "params": {"order_id": "ORD-7763"}}
RESULT:   ✅ Correct (lucky!)


### Observation

The 1B instruct model typically:
- Responds in natural language instead of JSON
- Adds preamble text before/after the JSON
- Gets the tool name wrong or hallucinates parameters
- Is inconsistent — sometimes close, sometimes garbage

**This is the problem we're solving with fine-tuning.**

---
## Step 3: Create the Training Dataset

We generate ~80 diverse examples — 40 per tool.
Each example is a conversation: system prompt → user query → correct JSON tool call.

**Key insight for your students:** The dataset IS the product. The model architecture is fixed, the training code is boilerplate. Your engineering judgment lives entirely in the data.

In [ ]:
import json
import random

# ──────────────────────────────────────────────
# TRACK_ORDER examples
# ──────────────────────────────────────────────
track_order_examples = [
    ("Where is my order ORD-1001?", {"order_id": "ORD-1001"}),
    ("Can you check the delivery status of ORD-2233?", {"order_id": "ORD-2233"}),
    ("I ordered something last week, order number ORD-5567. When will it arrive?", {"order_id": "ORD-5567"}),
    ("My order ORD-8899 hasn't been delivered yet. It's been 5 days.", {"order_id": "ORD-8899"}),
    ("Track my package please. Order ID is ORD-3344.", {"order_id": "ORD-3344"}),
    ("Has my shipment for ORD-7712 been dispatched?", {"order_id": "ORD-7712"}),
    ("I need to know where ORD-4421 is right now.", {"order_id": "ORD-4421"}),
    ("Any update on order ORD-6650? I'm waiting since Monday.", {"order_id": "ORD-6650"}),
    ("Please tell me the shipping status of ORD-1199.", {"order_id": "ORD-1199"}),
    ("Is ORD-8801 out for delivery today?", {"order_id": "ORD-8801"}),
    ("I want to track ORD-3009. Where is it stuck?", {"order_id": "ORD-3009"}),
    ("My friend ordered for me. The order number is ORD-2277. Can you check?", {"order_id": "ORD-2277"}),
    ("ORD-5543 — is this in transit or still being packed?", {"order_id": "ORD-5543"}),
    ("How long until ORD-9902 reaches me?", {"order_id": "ORD-9902"}),
    ("Delivery for ORD-1123 was supposed to be yesterday. What happened?", {"order_id": "ORD-1123"}),
    ("Did ORD-6677 ship yet?", {"order_id": "ORD-6677"}),
    ("Can I get a tracking link for ORD-4489?", {"order_id": "ORD-4489"}),
    ("I placed order ORD-3388 two weeks ago. Still no update.", {"order_id": "ORD-3388"}),
    ("What courier is delivering ORD-7766?", {"order_id": "ORD-7766"}),
    ("ORD-5512 says delivered but I didn't receive it.", {"order_id": "ORD-5512"}),
    ("The tracking page for ORD-2201 isn't loading. Can you check from your end?", {"order_id": "ORD-2201"}),
    ("Hey just checking on ORD-8834, any movement?", {"order_id": "ORD-8834"}),
    ("When was ORD-1155 dispatched?", {"order_id": "ORD-1155"}),
    ("I see ORD-9943 is in Bangalore hub. When will it come to Hyderabad?", {"order_id": "ORD-9943"}),
    ("Order status for ORD-6230 please.", {"order_id": "ORD-6230"}),
    ("Still waiting for ORD-7188. It's urgent.", {"order_id": "ORD-7188"}),
    ("Check ORD-4455 — my wife is asking about it.", {"order_id": "ORD-4455"}),
    ("Has ORD-3667 cleared customs yet?", {"order_id": "ORD-3667"}),
    ("ORD-8102. Where. Is. It.", {"order_id": "ORD-8102"}),
    ("I need delivery confirmation for ORD-5079.", {"order_id": "ORD-5079"}),
    ("Can you look up ORD-2918? I haven't gotten any shipping notification.", {"order_id": "ORD-2918"}),
    ("What is the estimated delivery date for ORD-4600?", {"order_id": "ORD-4600"}),
    ("ORD-7331 — the app says processing since 3 days.", {"order_id": "ORD-7331"}),
    ("Please track order ORD-1844 for me.", {"order_id": "ORD-1844"}),
    ("Any idea why ORD-9215 is delayed?", {"order_id": "ORD-9215"}),
    ("I want a refund if ORD-6503 doesn't arrive today. But first tell me where it is.", {"order_id": "ORD-6503"}),
    ("Bro where is ORD-3790, I ordered it ages ago", {"order_id": "ORD-3790"}),
    ("Just tell me if ORD-8047 has shipped or not.", {"order_id": "ORD-8047"}),
    ("ORD-5281 tracking shows stuck at warehouse. Why?", {"order_id": "ORD-5281"}),
    ("Update on ORD-2466? Need it before the weekend.", {"order_id": "ORD-2466"}),
]

# ──────────────────────────────────────────────
# SEARCH_PRODUCTS examples
# ──────────────────────────────────────────────
search_product_examples = [
    ("I need a good laptop for programming under 60000", {"query": "laptop for programming", "max_budget": 60000}),
    ("Show me noise cancelling headphones", {"query": "noise cancelling headphones", "max_budget": None}),
    ("What wireless mice do you have under 2000?", {"query": "wireless mouse", "max_budget": 2000}),
    ("I want to buy a mechanical keyboard for gaming", {"query": "mechanical keyboard gaming", "max_budget": None}),
    ("Suggest some monitors for coding. Budget 25000.", {"query": "monitor for coding", "max_budget": 25000}),
    ("Any good webcams for video calls?", {"query": "webcam video calls", "max_budget": None}),
    ("Looking for an ergonomic office chair under 15000", {"query": "ergonomic office chair", "max_budget": 15000}),
    ("What are the best tablets for reading? Don't want to spend more than 30000", {"query": "tablet for reading", "max_budget": 30000}),
    ("I need a USB-C hub. What options do you have?", {"query": "USB-C hub", "max_budget": None}),
    ("Recommend a portable SSD, at least 1TB, under 8000", {"query": "portable SSD 1TB", "max_budget": 8000}),
    ("Do you sell standing desks? Budget around 20000.", {"query": "standing desk", "max_budget": 20000}),
    ("What's a good phone for my mom? She just needs WhatsApp and camera. Under 12000.", {"query": "phone basic camera WhatsApp", "max_budget": 12000}),
    ("I want a 4K monitor for photo editing", {"query": "4K monitor photo editing", "max_budget": None}),
    ("Best earbuds under 3000?", {"query": "earbuds", "max_budget": 3000}),
    ("Show me gaming laptops with RTX 4060", {"query": "gaming laptop RTX 4060", "max_budget": None}),
    ("I need a printer for home use. Nothing fancy, under 10000.", {"query": "home printer", "max_budget": 10000}),
    ("Any Bluetooth speakers in the 5000-8000 range?", {"query": "Bluetooth speaker", "max_budget": 8000}),
    ("What drawing tablets do you recommend for beginners?", {"query": "drawing tablet beginner", "max_budget": None}),
    ("Looking for a NAS for home media server. Budget 40000.", {"query": "NAS home media server", "max_budget": 40000}),
    ("Suggest some good power banks. Need at least 20000mAh.", {"query": "power bank 20000mAh", "max_budget": None}),
    ("Compare your top 3 laptops under 80000 for software development", {"query": "laptop software development", "max_budget": 80000}),
    ("What routers are good for a 3BHK apartment?", {"query": "router large apartment", "max_budget": None}),
    ("I want to buy a smartwatch. Something sporty under 10000.", {"query": "smartwatch sporty", "max_budget": 10000}),
    ("Any recommendations for a budget graphic card? Under 25000.", {"query": "graphics card budget", "max_budget": 25000}),
    ("Need a microphone for podcasting", {"query": "microphone podcasting", "max_budget": None}),
    ("Show me your cheapest laptops", {"query": "cheapest laptop", "max_budget": None}),
    ("What RAM sticks are compatible with a Dell Inspiron?", {"query": "RAM Dell Inspiron compatible", "max_budget": None}),
    ("I want a good mouse pad, large size, under 1500", {"query": "large mouse pad", "max_budget": 1500}),
    ("Suggest some good study lamps with USB charging port", {"query": "study lamp USB charging", "max_budget": None}),
    ("Best laptop cooling pads under 2000?", {"query": "laptop cooling pad", "max_budget": 2000}),
    ("Looking for a DSLR camera for beginners. Budget is 50000.", {"query": "DSLR camera beginner", "max_budget": 50000}),
    ("What's a solid keyboard mouse combo under 3000?", {"query": "keyboard mouse combo", "max_budget": 3000}),
    ("Recommend an air purifier for a small room", {"query": "air purifier small room", "max_budget": None}),
    ("Show me laptop bags that fit a 15.6 inch screen", {"query": "laptop bag 15.6 inch", "max_budget": None}),
    ("What UPS should I get for a gaming PC? Budget 6000.", {"query": "UPS gaming PC", "max_budget": 6000}),
    ("Any good monitor arms? I want a dual monitor setup.", {"query": "dual monitor arm", "max_budget": None}),
    ("Need a fast charger for iPhone. What's available?", {"query": "fast charger iPhone", "max_budget": None}),
    ("Which e-readers do you have in stock?", {"query": "e-reader", "max_budget": None}),
    ("I want to upgrade my PC's SSD. 500GB NVMe under 4000.", {"query": "NVMe SSD 500GB", "max_budget": 4000}),
    ("What are your best-selling products this month?", {"query": "best selling products", "max_budget": None}),
]

print(f"track_order examples:   {len(track_order_examples)}")
print(f"search_products examples: {len(search_product_examples)}")
print(f"Total: {len(track_order_examples) + len(search_product_examples)}")

track_order examples:   40
search_products examples: 40
Total: 80


In [ ]:
# Convert to the conversations format that SFTTrainer expects

def make_conversation(user_query, tool_name, params):
    tool_call = json.dumps({"tool": tool_name, "params": params})
    return {
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_query},
            {"role": "assistant", "content": tool_call},
        ]
    }

dataset_rows = []

for query, params in track_order_examples:
    dataset_rows.append(make_conversation(query, "track_order", params))

for query, params in search_product_examples:
    dataset_rows.append(make_conversation(query, "search_products", params))

# Shuffle so the model doesn't just learn "first half = tool A"
random.seed(42)
random.shuffle(dataset_rows)

# Let's look at a few examples
print("Sample training conversation:")
print(json.dumps(dataset_rows[0], indent=2))
print("\n" + "─" * 60)
print(json.dumps(dataset_rows[1], indent=2))

Sample training conversation:
{
  "conversations": [
    {
      "role": "system",
      "content": "You are a customer support assistant. You have access to exactly two tools:\n\n1. track_order: Use when the customer asks about order status, shipping, delivery, or tracking.\n   Parameters: {\"order_id\": string}\n\n2. search_products: Use when the customer asks about finding products, recommendations, comparisons, or availability.\n   Parameters: {\"query\": string, \"max_budget\": number or null}\n\nRESPOND WITH ONLY a JSON object in this exact format, nothing else:\n{\"tool\": \"<tool_name>\", \"params\": {<parameters>}}\n\nDo not explain. Do not add any text before or after the JSON."
    },
    {
      "role": "user",
      "content": "Recommend a portable SSD, at least 1TB, under 8000"
    },
    {
      "role": "assistant",
      "content": "{\"tool\": \"search_products\", \"params\": {\"query\": \"portable SSD 1TB\", \"max_budget\": 8000}}"
    }
  ]
}

────────────────────────

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(dataset_rows)
print(dataset)
print(f"\nTotal training examples: {len(dataset)}")

Dataset({
    features: ['conversations'],
    num_rows: 80
})

Total training examples: 80


---
## Step 4: Attach LoRA Adapters

Instead of updating all 1B parameters, LoRA trains two small matrices (rank 16) on each attention and MLP layer.

**Why this matters:** A 1B model has ~1 billion weights. With LoRA rank=16, we only train ~10 million weights (1%). This means:
- 50-70% less VRAM
- 2x faster training
- The base model knowledge is preserved — we're just teaching it a new output format

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                    # LoRA rank — 16 is a good default
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention layers
        "gate_proj", "up_proj", "down_proj",       # MLP layers
    ],
    lora_alpha = 16,           # Scaling factor of lora updates: If you set alpha=32 with r=16, the LoRA updates are multiplied by 2x. The model learns faster but can overshoot and destabilize.
    lora_dropout = 0,          # how many neurons are ignored
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # 30% less VRAM
    random_state = 42,
    max_seq_length = 2048,
)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

Unsloth 2026.4.2 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Trainable parameters: 11,272,192 / 760,547,328 (1.48%)


---
## Step 5: Fine-Tune!

We train for **3 epochs** over 80 examples = 240 training steps.
On a T4, this takes roughly **5–10 minutes**.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
formatted_rows = []
for row in dataset_rows:
    text = tokenizer.apply_chat_template(
        row["conversations"],
        tokenize=False,
        add_generation_prompt=False,
    )
    formatted_rows.append({"text": text})

dataset = Dataset.from_list(formatted_rows)

# Check what the training text actually looks like
print(dataset[0]["text"][:500])
print("─" * 60)
print(f"Total training examples: {len(dataset)}")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",         # <-- just point to the pre-formatted column
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        num_train_epochs = 3,
        warmup_steps = 10,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "tool-routing-checkpoints",
        max_seq_length = 2048,
    ),
)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 05 Apr 2026

You are a customer support assistant. You have access to exactly two tools:

1. track_order: Use when the customer asks about order status, shipping, delivery, or tracking.
   Parameters: {"order_id": string}

2. search_products: Use when the customer asks about finding products, recommendations, comparisons, or availability.
   Parameters: {"query": string, "max_budget": n
────────────────────────────────────────────────────────────
Total training examples: 80


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/80 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
print("Starting training...")
trainer_stats = trainer.train()

print(f"\n{'=' * 50}")
print(f"Training complete!")
print(f"Total time:   {trainer_stats.metrics['train_runtime']:.0f} seconds")
print(f"Final loss:   {trainer_stats.metrics['train_loss']:.4f}")
print(f"{'=' * 50}")

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 80 | Num Epochs = 3 | Total steps = 30
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,3.310673
10,2.381383
15,1.178822
20,0.474263
25,0.373597
30,0.358585



Training complete!
Total time:   51 seconds
Final loss:   1.3462


---
## Step 6: Test the Fine-Tuned Model

Now the moment of truth. We run the **exact same queries** from Step 2.

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

print("=" * 70)
print("FINE-TUNED MODEL RESPONSES")
print("=" * 70)

correct = 0
for query, expected_tool in zip(TEST_QUERIES, EXPECTED):
    response = run_inference(model, tokenizer, query)
    print(f"\n{'─' * 60}")
    print(f"QUERY:    {query[:80]}")
    print(f"EXPECTED: {expected_tool}")
    print(f"GOT:      {response[:200]}")

    try:
        parsed = json.loads(response)
        if parsed.get("tool") == expected_tool:
            print("RESULT:   ✅ Correct tool + valid JSON")
            correct += 1
        else:
            print(f"RESULT:   ❌ Wrong tool (got '{parsed.get('tool')}')'")
    except json.JSONDecodeError:
        print("RESULT:   ❌ Not valid JSON")

print(f"\n{'=' * 70}")
print(f"Score: {correct}/{len(TEST_QUERIES)} correct")
print(f"{'=' * 70}")

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FINE-TUNED MODEL RESPONSES


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    I want to order a new gaming mouse. What do you have?
EXPECTED: search_products
GOT:      {"tool": "search_products", "params": {"query": "gaming mouse", "max_budget": null}}
RESULT:   ✅ Correct tool + valid JSON


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


────────────────────────────────────────────────────────────
QUERY:    bro just tell me if ORD-2290 is coming today or not
EXPECTED: track_order
GOT:      {"tool": "track_order", "params": {"order_id": "ORD-2290"}}
RESULT:   ✅ Correct tool + valid JSON


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    my budget is 10000, surprise me with something cool
EXPECTED: search_products
GOT:      {"tool": "search_products", "params": {"query": "cool", "max_budget": 10000}}
RESULT:   ✅ Correct tool + valid JSON


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    I saw a monitor listed at 24999 on your site. Got anything similar but cheaper?
EXPECTED: search_products
GOT:      {"tool": "search_products", "params": {"query": "monitor", "max_budget": 24999}}
RESULT:   ✅ Correct tool + valid JSON

────────────────────────────────────────────────────────────
QUERY:    So basically my wife's birthday is tomorrow and I ordered her gift last week, or
EXPECTED: track_order
GOT:      {"tool": "track_order", "params": {"order_id": "ORD-7763"}}
RESULT:   ✅ Correct tool + valid JSON

Score: 10/10 correct


---
## Step 7: Try Completely New Queries

These are queries the model has **never seen** — testing generalization.

In [ ]:
NEW_QUERIES = [
    # Should → track_order
    ("My parcel ORD-7777 seems lost. Help!", "track_order"),
    ("ORD-4040 — shipped or not?", "track_order"),
    ("Can you ping the courier about ORD-6161?", "track_order"),

    # Should → search_products
    ("What's a good gift for a programmer? Budget 5000.", "search_products"),
    ("Do you have any wireless chargers?", "search_products"),
    ("I need to set up a home office. What all do I need under 50000?", "search_products"),
]

print("=" * 70)
print("GENERALIZATION TEST — QUERIES NEVER SEEN IN TRAINING")
print("=" * 70)

correct = 0
for query, expected_tool in NEW_QUERIES:
    response = run_inference(model, tokenizer, query)
    print(f"\n{'─' * 60}")
    print(f"QUERY:    {query}")
    print(f"EXPECTED: {expected_tool}")
    print(f"GOT:      {response[:200]}")

    try:
        parsed = json.loads(response)
        tool_correct = parsed.get("tool") == expected_tool
        has_params = "params" in parsed
        if tool_correct and has_params:
            print(f"RESULT:   ✅ Correct tool + valid params")
            correct += 1
        elif tool_correct:
            print(f"RESULT:   ⚠️  Correct tool but missing params")
        else:
            print(f"RESULT:   ❌ Wrong tool")
    except json.JSONDecodeError:
        print(f"RESULT:   ❌ Not valid JSON")

print(f"\n{'=' * 70}")
print(f"Generalization score: {correct}/{len(NEW_QUERIES)} correct")
print(f"{'=' * 70}")

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERALIZATION TEST — QUERIES NEVER SEEN IN TRAINING


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    My parcel ORD-7777 seems lost. Help!
EXPECTED: track_order
GOT:      {"tool": "track_order", "params": {"order_id": "ORD-7777"}}
RESULT:   ✅ Correct tool + valid params


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


────────────────────────────────────────────────────────────
QUERY:    Can you ping the courier about ORD-6161?
EXPECTED: track_order
GOT:      {"tool": "track_order", "params": {"order_id": "ORD-6161"}}
RESULT:   ✅ Correct tool + valid params


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    What's a good gift for a programmer? Budget 5000.
EXPECTED: search_products
GOT:      {"tool": "search_products", "params": {"query": "programming gifts", "max_budget": 5000}}
RESULT:   ✅ Correct tool + valid params


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────
QUERY:    Do you have any wireless chargers?
EXPECTED: search_products
GOT:      {"tool": "search_products", "params": {"query": "wireless charger", "max_budget": null}}
RESULT:   ✅ Correct tool + valid params

────────────────────────────────────────────────────────────
QUERY:    I need to set up a home office. What all do I need under 50000?
EXPECTED: search_products
GOT:      {"tool": "search_products", "params": {"query": "home office setup", "max_budget": 50000}}
RESULT:   ✅ Correct tool + valid params

Generalization score: 6/6 correct


---
## Step 8: Save & Export

Export to GGUF for local deployment with Ollama or llama.cpp.

In [ ]:
# Save LoRA adapters (small, ~30MB)
model.save_pretrained("tool-router-lora")
tokenizer.save_pretrained("tool-router-lora")
print("LoRA adapters saved to tool-router-lora/")

# Optional: Export merged GGUF for local deployment
# Uncomment the line below if you want to run this with Ollama
# model.save_pretrained_gguf("tool-router-gguf", tokenizer, quantization_method="q4_k_m")
# print("GGUF exported to tool-router-gguf/")

LoRA adapters saved to tool-router-lora/


---
## Recap

| Step | What happened |
|---|---|
| Base model test | 1B model failed at structured JSON tool routing |
| Dataset | 80 examples, 40 per tool, manually crafted |
| Fine-tuning | LoRA rank 16, 3 epochs, ~5-10 min on T4 |
| Result | Model correctly routes to `track_order` vs `search_products` |
| Generalization | Works on completely unseen queries |

### Key Takeaways

1. **Data > model size.** 80 examples + LoRA on 1B params beat a generic 8B model at this task.
2. **Fine-tuning teaches FORMAT, not knowledge.** The model already knew about orders and products. We taught it to express that knowledge as structured JSON tool calls.
3. **Evaluation must be binary.** Tool-calling has a clear pass/fail: is the JSON valid? Is the tool correct? Are the params right? This is infinitely better than vibes-based eval.
4. **This is how you build reliable agents.** An agent loop that calls tools needs a model that produces valid tool calls 99%+ of the time. Fine-tuning gets you there. Prompting alone doesn't.

### What to try next
- Add a 3rd tool (e.g., `cancel_order`) and retrain — see how few examples you need
- Try ambiguous queries that could go either way
- Build an agent loop that actually executes the tool calls
- Export to GGUF and deploy locally with Ollama